[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_11_12/05_tag_11_12_rnn_ein_und_ausgabeformen.ipynb)

# Tag 11 & 12 – 05: Sequence-to-Vector und Sequence-to-Sequence

Ein RNN verarbeitet immer eine Sequenz von Eingabewerten. Die Ausgabe kann jedoch entweder eine einzelne Entscheidung für die gesamte Sequenz oder ein Wert pro Zeitschritt sein. Dieses Notebook verwendet denselben Datensatz für beide Fälle.

## Zwei Aufgaben, dieselben Eingabesequenzen

Jede Sequenz enthält 40 verrauschte Werte einer periodischen Kurve. Es gibt zwei Klassen: eine langsame und eine schnelle Schwingung.

| Aufgabe | Eingabe | Ziel | Ausgabeform |
| --- | --- | --- | --- |
| Sequence-to-Vector | ganze verrauschte Sequenz | Klasse `langsam` oder `schnell` | eine Zahl pro Sequenz |
| Sequence-to-Sequence | ganze verrauschte Sequenz | sauberer Wert zu jedem Zeitschritt | ein Wert pro Zeitschritt |

Die Eingabeform ist für beide Modelle identisch: `(Anzahl Sequenzen, 40 Zeitschritte, 1 Merkmal)`. Entscheidend ist die Form des Ziel-Tensors und ob die RNN-Schicht nur den letzten oder alle Hidden States ausgibt.

In [ ]:
import matplotlib.pyplot as plt  # Diagramme für Daten und Modellresultate
import numpy as np               # Arrays und künstliche Sequenzen
import tensorflow as tf          # TensorFlow-Modelle mit SimpleRNN

plt.style.use('seaborn-v0_8-whitegrid')  # Einheitliches Darstellungsformat
RANDOM_STATE = 7                         # Fester Zufallsstart für reproduzierbare Ergebnisse
rng = np.random.default_rng(RANDOM_STATE)  # NumPy-Zufallsgenerator erzeugen
tf.keras.utils.set_random_seed(RANDOM_STATE)  # TensorFlow-Gewichte reproduzierbar initialisieren

## Derselbe Datensatz mit zwei Zieldefinitionen

Die Klasse beschreibt die Frequenz der gesamten Sequenz. Der Sequenz-Zielwert enthält dagegen die rauschfreie Kurve an jeder Position. Damit lässt sich an exakt denselben Eingabewerten sowohl Klassifikation als auch zeitpunktweise Rekonstruktion zeigen.

In [ ]:
ANZAHL_SEQUENZEN = 480  # Anzahl unabhängiger Beispiele
SEQUENZLAENGE = 40      # Anzahl Werte pro Beispiel
TRAINING_ANTEIL = 0.75  # Anteil der Sequenzen für das Training

positionen = np.arange(SEQUENZLAENGE)  # Zeitachse innerhalb einer einzelnen Sequenz
X = np.zeros((ANZAHL_SEQUENZEN, SEQUENZLAENGE, 1), dtype=np.float32)  # Eingabetensor vorbereiten
y_klasse = np.zeros(ANZAHL_SEQUENZEN, dtype=np.float32)                # Ein Klassenlabel pro Sequenz vorbereiten
y_folge = np.zeros((ANZAHL_SEQUENZEN, SEQUENZLAENGE, 1), dtype=np.float32)  # Ein Zielwert pro Zeitschritt vorbereiten

for nummer in range(ANZAHL_SEQUENZEN):  # Alle künstlichen Sequenzen nacheinander erzeugen
    klasse = nummer % 2  # Klasse 0 und 1 abwechselnd erzeugen, damit der Datensatz ausgeglichen ist
    periode = 24 if klasse == 0 else 8  # Langsame Klasse hat Periode 24, schnelle Klasse Periode 8
    phase = rng.uniform(0, 2 * np.pi)  # Jede Sequenz startet an einer zufälligen Phase
    amplitude = rng.uniform(0.8, 1.2)  # Die Amplitude variiert leicht zwischen den Beispielen
    saubere_folge = amplitude * np.sin(2 * np.pi * positionen / periode + phase)  # Rauschfreies Ziel erzeugen
    verrauschte_folge = saubere_folge + rng.normal(0, 0.20, SEQUENZLAENGE)  # Beobachtete Eingabe verrauschen
    X[nummer, :, 0] = verrauschte_folge  # Verrauschte Werte als RNN-Eingabe speichern
    y_klasse[nummer] = klasse            # Eine Klasse für die gesamte Sequenz speichern
    y_folge[nummer, :, 0] = saubere_folge  # Sauberen Zielwert an jeder Position speichern

reihenfolge = rng.permutation(ANZAHL_SEQUENZEN)  # Beispiele zufällig mischen
train_grenze = int(ANZAHL_SEQUENZEN * TRAINING_ANTEIL)  # Position zwischen Training und Test bestimmen
train_indizes = reihenfolge[:train_grenze]  # Indizes der Trainingssequenzen auswählen
test_indizes = reihenfolge[train_grenze:]   # Indizes der Testsequenzen auswählen

X_train, X_test = X[train_indizes], X[test_indizes]  # Eingaben in Training und Test aufteilen
y_klasse_train, y_klasse_test = y_klasse[train_indizes], y_klasse[test_indizes]  # Klassenlabels aufteilen
y_folge_train, y_folge_test = y_folge[train_indizes], y_folge[test_indizes]  # Sequenzziele aufteilen

print('Eingabe X:', X.shape)  # Gemeinsame Eingabeform ausgeben
print('Sequence-to-Vector-Ziel y_klasse:', y_klasse.shape)  # Eine Zahl pro Sequenz zeigen
print('Sequence-to-Sequence-Ziel y_folge:', y_folge.shape)  # 40 Zahlen pro Sequenz zeigen

beispiel_indizes = [0, 1, 2, 3]  # Zwei langsame und zwei schnelle Sequenzen für den ersten Überblick auswählen
fig, achsen = plt.subplots(2, 2, figsize=(12, 6), sharex=True, sharey=True)  # Vier Beispiele als Raster anordnen
for achse, beispiel in zip(achsen.ravel(), beispiel_indizes):  # Jede Achse mit einer Sequenz befüllen
    klassenname = 'langsam' if y_klasse[beispiel] == 0 else 'schnell'  # Numerisches Label in einen lesbaren Namen übersetzen
    achse.plot(positionen, X[beispiel, :, 0], 'o-', color='0.55', markersize=3, label='verrauschte Eingabe')  # RNN-Eingabe zeichnen
    achse.plot(positionen, y_folge[beispiel, :, 0], color='tab:green', linewidth=2, label='sauberes Ziel')  # Saubere Zielkurve zeichnen
    achse.set_title(f'Beispiel {beispiel}: Klasse {klassenname}')  # Klasse im jeweiligen Teilplot benennen
    achse.set_xlabel('Zeitschritt')  # x-Achse des Teilplots beschriften
    achse.set_ylabel('Messwert')     # y-Achse des Teilplots beschriften
achsen[0, 0].legend(loc='upper right')  # Eine gemeinsame Legende im ersten Teilplot anzeigen
fig.suptitle('Ein Datensatz, zwei Klassen und zusätzlich ein sauberer Zielwert pro Zeitschritt')  # Gesamttitel setzen
plt.tight_layout()  # Abstände zwischen Teilplots anpassen
plt.show()          # Abbildung ausgeben

## Sequence-to-Vector: Eine Klasse pro kompletter Sequenz

Hier liest das RNN alle 40 Werte. `return_sequences=False` bedeutet: Nur der letzte Hidden State wird an den Dense-Layer weitergegeben. Die Ausgabeform ist daher `(Batchgröße, 1)` – eine Wahrscheinlichkeit für die Klasse `schnell` pro Sequenz.

In [ ]:
eingabe_vector = tf.keras.Input(shape=(SEQUENZLAENGE, 1), name='sequenz')  # 40 Werte mit je einem Merkmal
letzter_state = tf.keras.layers.SimpleRNN(12, return_sequences=False, name='rnn_vector')(eingabe_vector)  # Nur h(40) ausgeben
klassen_wahrscheinlichkeit = tf.keras.layers.Dense(1, activation='sigmoid', name='klasse')(letzter_state)  # Aus h(40) eine Klassenwahrscheinlichkeit machen
modell_vector = tf.keras.Model(eingabe_vector, klassen_wahrscheinlichkeit, name='sequence_to_vector')  # Komplettes Klassifikationsmodell bilden

modell_vector.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])  # Binäre Klassifikation konfigurieren
historie_vector = modell_vector.fit(X_train, y_klasse_train, validation_split=0.15, shuffle=False, epochs=18, batch_size=32, verbose=0)  # Modell trainieren
test_loss_vector, test_accuracy = modell_vector.evaluate(X_test, y_klasse_test, verbose=0)  # Klassifikation auf Testsequenzen bewerten

modell_vector.summary()  # Architektur und Outputform anzeigen
print()  # Leerzeile zur besseren Lesbarkeit
print(f'Test-Genauigkeit Sequence-to-Vector: {test_accuracy:.3f}')  # Testgenauigkeit ausgeben

## Sequence-to-Sequence: Ein Zielwert pro Zeitschritt

Hier muss das RNN an jeder der 40 Positionen einen Wert ausgeben. Deshalb bleibt `return_sequences=True`: Die RNN-Schicht liefert eine Folge von 40 Hidden States. Ein `TimeDistributed(Dense(1))` wandelt jeden dieser States in einen bereinigten Messwert um. Die Ausgabeform ist `(Batchgröße, 40, 1)`.

In [ ]:
eingabe_folge = tf.keras.Input(shape=(SEQUENZLAENGE, 1), name='sequenz')  # Dieselbe Eingabeform wie beim Klassifikationsmodell
alle_states = tf.keras.layers.SimpleRNN(12, return_sequences=True, name='rnn_folge')(eingabe_folge)  # h(1) bis h(40) zurückgeben
bereinigte_folge = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(1), name='wert_pro_zeitpunkt')(alle_states)  # Jeden Hidden State in einen Messwert übersetzen
modell_folge = tf.keras.Model(eingabe_folge, bereinigte_folge, name='sequence_to_sequence')  # Komplettes Rekonstruktionsmodell bilden

modell_folge.compile(optimizer='adam', loss='mse', metrics=['mae'])  # Punktweise Regression mit MSE trainieren
historie_folge = modell_folge.fit(X_train, y_folge_train, validation_split=0.15, shuffle=False, epochs=24, batch_size=32, verbose=0)  # Modell trainieren
test_loss_folge, test_mae = modell_folge.evaluate(X_test, y_folge_test, verbose=0)  # Mittleren Fehler auf Testsequenzen bestimmen

modell_folge.summary()  # Architektur und die zeitliche Outputform anzeigen
print()  # Leerzeile zur besseren Lesbarkeit
print(f'Test-MAE Sequence-to-Sequence: {test_mae:.3f}')  # Fehler pro Zeitschritt ausgeben

## Beide Ausgabeformen am gleichen Testbeispiel

Links steht die einzelne Klassenentscheidung für die gesamte Sequenz. Rechts sieht man die 40 zeitpunktweisen Ausgaben des Rekonstruktionsmodells. Die Eingabe ist in beiden Fällen exakt dieselbe.

In [ ]:
BEISPIEL_NUMMER = 3  # Ein Testbeispiel für beide Modellarten auswählen
beispiel_x = X_test[BEISPIEL_NUMMER : BEISPIEL_NUMMER + 1]  # Batch mit einer Eingabesequenz erstellen
beispiel_klasse = int(y_klasse_test[BEISPIEL_NUMMER])  # Wahre Klasse der Sequenz lesen
beispiel_folge = y_folge_test[BEISPIEL_NUMMER, :, 0]  # Saubere Zielkurve der Sequenz lesen

klasse_wahrscheinlichkeit = modell_vector.predict(beispiel_x, verbose=0)[0, 0]  # Eine einzige Wahrscheinlichkeit für die ganze Sequenz erzeugen
klasse_prognose = int(klasse_wahrscheinlichkeit >= 0.5)  # Wahrscheinlichkeit mit 0.5 in eine Klasse übersetzen
folge_prognose = modell_folge.predict(beispiel_x, verbose=0)[0, :, 0]  # 40 vorhergesagte Werte erzeugen

fig, (ax_klasse, ax_folge) = plt.subplots(1, 2, figsize=(13, 4))  # Zwei nebeneinanderliegende Diagramme erzeugen
balken = ax_klasse.bar(['langsam', 'schnell'], [1 - klasse_wahrscheinlichkeit, klasse_wahrscheinlichkeit], color=['tab:blue', 'tab:orange'])  # Klassenwahrscheinlichkeiten zeichnen
ax_klasse.bar_label(balken, fmt='%.2f', padding=3)  # Zahlen direkt über den Balken anzeigen
ax_klasse.set_ylim(0, 1.12)  # Wahrscheinlichkeitsskala auf 0 bis 1 begrenzen
titel_klasse = f'Sequence-to-Vector: vorhergesagt = {"langsam" if klasse_prognose == 0 else "schnell"} | wahre Klasse = {"langsam" if beispiel_klasse == 0 else "schnell"}'  # Titeltext zusammensetzen
ax_klasse.set(title=titel_klasse, ylabel='Wahrscheinlichkeit')  # Entscheidung beschriften

ax_folge.plot(positionen, beispiel_x[0, :, 0], 'o-', color='0.55', label='verrauschte Eingabe')  # Gemeinsame Eingabe zeichnen
ax_folge.plot(positionen, beispiel_folge, color='tab:green', linewidth=2, label='sauberes Ziel')  # Wahrer Wert pro Zeitschritt
ax_folge.plot(positionen, folge_prognose, color='tab:orange', linestyle='--', linewidth=2, label='Sequence-to-Sequence-Ausgabe')  # Modellwert pro Zeitschritt
ax_folge.set(title='Sequence-to-Sequence: 40 vorhergesagte Werte', xlabel='Zeitschritt', ylabel='Messwert')  # Achsen beschriften
ax_folge.legend(loc='upper right')  # Legende anzeigen
plt.tight_layout()  # Abstände der beiden Diagramme anpassen
plt.show()          # Abbildung ausgeben

print('Eingabeform beider Modelle:', beispiel_x.shape)  # Form eines einzelnen Eingabebatches zeigen
print('Outputform Sequence-to-Vector:', modell_vector.output_shape)  # Eine Zahl pro Sequenz zeigen
print('Outputform Sequence-to-Sequence:', modell_folge.output_shape)  # Eine Zahl pro Zeitschritt zeigen

## Kerngedanke

Die Zeitachse der Eingabe bleibt in beiden Fällen gleich. Bei Sequence-to-Vector wird die gesamte Sequenz im letzten Hidden State verdichtet und führt zu einer Entscheidung. Bei Sequence-to-Sequence bleibt die Zeitachse bis zur Ausgabe erhalten: Jeder Hidden State erzeugt einen eigenen Zielwert.